# 🚇 Metro-ASR — Streaming & Server Deployment

This notebook shows how to:
1. Deploy Metro-ASR as a REST API server
2. Send requests via Python, curl, and Postman
3. Use streaming transcription for real-time audio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammedAly22/Metro-ASR/blob/main/examples/streaming_server.ipynb)

## 📦 Installation

In [ ]:
!pip install metro-asr[server] -q

## 🖥️ Option 1: REST API Server

### Start the server
```bash
python scripts/serve.py
```

### Send requests

### Python Client

In [ ]:
import requests

SERVER_URL = "http://localhost:8000"

# Health check
resp = requests.get(f"{SERVER_URL}/health")
print(f"Server status: {resp.json()}")

# Model info
resp = requests.get(f"{SERVER_URL}/info")
print(f"Model info: {resp.json()}")

In [ ]:
# Transcribe a single file
with open("audio.wav", "rb") as f:
    resp = requests.post(
        f"{SERVER_URL}/transcribe",
        files={"audio": f},
    )

result = resp.json()
print(f"Text: {result['text']}")
print(f"Duration: {result['duration']}s")
print(f"RTF: {result['rtf']}")

In [ ]:
# Transcribe with beam search + LM
with open("audio.wav", "rb") as f:
    resp = requests.post(
        f"{SERVER_URL}/transcribe",
        files={"audio": f},
        data={"beam_search": "true"},
    )

print(f"Text: {resp.json()['text']}")

In [ ]:
# Batch transcription
files = [
    ("audio", open("audio1.wav", "rb")),
    ("audio", open("audio2.wav", "rb")),
]
resp = requests.post(f"{SERVER_URL}/transcribe/batch", files=files)

for r in resp.json()["results"]:
    print(f"  {r['text']}  ({r['duration']}s)")

### curl Examples

```bash
# Basic transcription
curl -X POST http://localhost:8000/transcribe \
    -F "audio=@audio.wav"

# With beam search
curl -X POST http://localhost:8000/transcribe \
    -F "audio=@audio.wav" \
    -F "beam_search=true"

# Batch
curl -X POST http://localhost:8000/transcribe/batch \
    -F "audio=@audio1.wav" \
    -F "audio=@audio2.wav"

# Health check
curl http://localhost:8000/health

# Model info
curl http://localhost:8000/info
```

### Postman

1. **Method**: `POST`
2. **URL**: `http://localhost:8000/transcribe`
3. **Body**: Select `form-data`
   - Key: `audio` (type: File), Value: Select your `.wav` file
   - Key: `beam_search` (type: Text), Value: `true` (optional)
4. Click **Send**

## 🔴 Option 2: Streaming Transcription (Python API)

In [ ]:
import numpy as np
import soundfile as sf
from metro_asr import MetroASREngine

engine = MetroASREngine.from_pretrained("small")

def audio_chunk_generator(audio_path, chunk_seconds=5.0):
    """Read audio file in chunks, simulating a microphone stream."""
    data, sr = sf.read(audio_path)
    chunk_size = int(sr * chunk_seconds)
    for i in range(0, len(data), chunk_size):
        yield data[i:i + chunk_size]

# Stream and get real-time transcription
for chunk in engine.transcribe_stream(audio_chunk_generator("long_audio.wav")):
    print(f"[{chunk.total_duration:.1f}s] {chunk.text}")
    if chunk.is_final:
        print("--- END ---")

## 🎮 Option 3: Gradio Streaming App

```bash
# Clone the repo
git clone https://github.com/MohammedAly22/Metro-ASR.git
cd Metro-ASR

# Install
pip install metro-asr[demo]

# Launch streaming app
python app_streaming.py
```

Opens a dark-mode web UI at `http://localhost:7861` with:
- 🎙 Live microphone streaming with real-time transcription
- 📁 File upload with beam search option
- 📊 Performance metrics (RTF, latency, speed)